# Bölüm 2 — Modelleme: Parametre Adından Kategori Tahmini

**Proje:** GPT Eklenti Ekosisteminde Gizlilik Riski Analizi ve Otomatik Sınıflandırma

**Araştırma Sorusu 3:** Sadece parametre adına (ve varsa açıklamasına) bakarak, o parametrenin hangi kategoriye ait olduğu ne doğrulukla tahmin edilebilir?

**Hedef değişken kararı (Bölüm 1 sonunda verildi):** Ana model `main_data_type` (25, dengeli ve yeterli örneklemli sınıf) üzerine kurulacak — sağlam bir baseline, güvenilir metrikler ve yorumlanabilir feature importance için. Bölüm 2'nin sonunda, aynı pipeline'ı nadir sınıfları gruplayarak `data_type` (145 ince sınıf) üzerinde de deneyeceğiz; bu ikinci deneme Bölüm 3'teki "Other" sınıflandırması (Araştırma Sorusu 5) için de temel oluşturacak.

**Bu bölümün planı:**
1. Problemi kurma: metin özelliği (`name` + `description`) ve hedef (`main_data_type`) tanımlama, train/test ayırma
2. TF-IDF + Lojistik Regresyon (baseline model)
3. Embedding tabanlı model
4. İki modelin karşılaştırılması, feature importance yorumu
5. Aynı pipeline'ın `data_type` (nadir sınıflar gruplanmış) üzerinde denenmesi

## Adım 1 — Problemi Kurma: Girdi/Çıktı Tanımı ve Train/Test Ayrımı

Bir sınıflandırma modeli kurmadan önce iki şeyi netleştirmemiz lazım: **modele ne veriyoruz (X)** ve **modelden ne bekliyoruz (y)**.

- **X (girdi metni):** `name` ve `description`'ı tek bir metinde birleştireceğiz. Bunun sebebi basit: kayıtların %15.6'sında `description` boş (Bölüm 1'de gördük), o zaman modelin elinde sadece `name` kalıyor. İkisini birleştirip tek bir metin sütunu yapmak hem eldeki bilgiyi kaybetmeden kullanmamızı sağlıyor hem de pipeline'ı sadeleştiriyor.
- **y (hedef):** `main_data_type` — 25 sınıf.

Sonra veriyi **train/test** olarak ikiye ayıracağız — modelin daha önce hiç görmediği kayıtlar üzerinde ne kadar başarılı olduğunu ölçebilmek için. Sınıflar dengesiz olduğunu biliyoruz (en büyüğü 2568, en küçüğü 26 kayıt); bu yüzden **stratified split** kullanacağız, yani her sınıfın train ve test kümelerinde yaklaşık aynı oranda temsil edilmesini garanti edeceğiz. Aksi halde küçük bir sınıfın tüm örnekleri şans eseri train'e (ya da test'e) gidebilir ve o sınıf için hiç sağlıklı ölçüm yapamayız.

In [1]:
import json
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

DATA_PATH = '../backend/data_entries_final.json'

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)
df['text'] = (df['name'].fillna('') + '. ' + df['description'].fillna('')).str.strip()

X = df['text']
y = df['main_data_type']

print('Toplam kayıt:', len(df))
print('Benzersiz sınıf sayısı:', y.nunique())
df[['name', 'description', 'text', 'main_data_type']].head(3)

Toplam kayıt: 12811
Benzersiz sınıf sayısı: 25


,name,description,text,main_data_type
0,version,The model version,version. The model version,App metadata
1,input,,input.,App usage data
2,prediction_id,,prediction_id.,Identifier


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train boyutu:', X_train.shape[0])
print('Test boyutu:', X_test.shape[0])
print()
print('En küçük sınıfın train/test dağılımı (Weather information):')
print('  train:', (y_train == 'Weather information').sum())
print('  test :', (y_test == 'Weather information').sum())

Train boyutu: 10248
Test boyutu: 2563

En küçük sınıfın train/test dağılımı (Weather information):
  train: 21
  test : 5
